# Base

In [7]:
import pandas as pd
import numpy as np
import datetime as dt

# ===============================================
import sys
from pathlib import Path

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))
# ===============================================


from pipeline.coleta import get_pmc_index, get_pmc_pesos
from sktime.transformations.hierarchical.aggregate import Aggregator

from pipeline.modelo import metrics
from pipeline.tratamento import date_to_period

import warnings
warnings.filterwarnings('ignore')


# Módulos skitme


In [8]:
# ==========
#  Pipeline
# ==========
from sktime.forecasting.compose import TransformedTargetForecaster, ForecastingPipeline

# =============================
#  Modelos univariados básicos
# =============================
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.statsforecast import (
                                                StatsForecastAutoARIMA, StatsForecastAutoETS, 
                                                StatsForecastAutoCES, StatsForecastAutoTBATS
                                            )

# ====================
#  Modelos compostos
# ====================
from sktime.forecasting.compose import AutoEnsembleForecaster

# ===============
#  Regressors
# ===============
from sklearn.linear_model import LinearRegression, ElasticNet
from lightgbm import LGBMRegressor, DaskLGBMRegressor
from catboost import CatBoostRegressor

# ============================
#  Métodos de reconciliação
# ============================
from sktime.forecasting.reconcile import ( 
                                            BottomUpReconciler, TopdownReconciler, 
                                            OptimalReconciler, ReconcilerForecaster
                                        )
# ===============
#  Transformers
# ===============
from sktime.transformations.series.boxcox import LogTransformer, BoxCoxTransformer
from sktime.transformations.series.detrend import Detrender, Deseasonalizer, ConditionalDeseasonalizer
from sktime.transformations.series.difference import Differencer
from sktime.transformations.compose import OptionalPassthrough

# ==================
#  Cross Validation
# ==================
from sktime.split import TemporalTrainTestSplitter, ExpandingWindowSplitter, SlidingWindowSplitter
from sktime.forecasting.model_evaluation import evaluate
from sktime.forecasting.model_selection import ForecastingOptunaSearchCV, ForecastingRandomizedSearchCV, ForecastingGridSearchCV
from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution

# ===========
#  Métricas
# ===========
from sktime.performance_metrics.forecasting import (
                                                    MeanAbsoluteError, MeanAbsoluteScaledError,
                                                    MeanAbsolutePercentageError, MeanSquaredError
                                                    )

# Coleta

In [9]:
from sktime.split import temporal_train_test_split

In [10]:
def date_to_period(df, date_col: str, multiindex: bool = True):
    df_fmt = (
            df
            .reset_index()
            .assign(
                    Data = lambda df: pd.PeriodIndex.from_fields(
                                                                month=df[date_col].dt.month, 
                                                                year=df[date_col].dt.year, 
                                                                freq='M'
                                                                )

                    )
            .set_index(list(df.index.names))
        )

    return df_fmt

In [11]:
pmc_raw = get_pmc_index('restrita_sem_aberturas').dropna().pipe(date_to_period, date_col='Data')

pesos_raw = get_pmc_pesos('restrita_sem_aberturas')
pesos_raw


pmc_agg = pmc_raw \
    .reset_index() \
    .merge(pesos_raw, on='Atividades', how='left') \
    .assign(indice_pond = lambda df: df['nindice'] * df['Pesos']/100) \
    .groupby(['Atividades', 'Data'])[['indice_pond']].last() \
    .pipe( Aggregator().fit_transform )

test_size=24
fh =  range(1, test_size + 1)
train, test = temporal_train_test_split(y=pmc_agg, test_size = test_size)

# Fit

In [ ]:
arima = StatsForecastAutoARIMA(sp=12)
tbats = StatsForecastAutoTBATS(seasonal_periods=12)
ets =  StatsForecastAutoETS(season_length=12)
ces = StatsForecastAutoCES(season_length=12)
snaive = NaiveForecaster(sp=12)
lgbm = LGBMRegressor(verbosity=-1)
ensemble = AutoEnsembleForecaster(forecasters=[arima, tbats, ets, ces, snaive], regressor=lgbm)

## Bottom-Up

In [27]:
bu_pipe = TransformedTargetForecaster(steps=[
    # ('log', LogTransformer()),
    ('deseason', Deseasonalizer(sp=12)),
    ('diff', Differencer()),
    ('forecaster', ensemble),
    ('reconciler', BottomUpReconciler())

])

bu_pipe.fit(train, fh=fh)
bu_preds = bu_pipe.predict()

bu_preds

indice_pond
Atividades                                   Data                
1. Combustíveis e lubrificantes              2024-02    11.485453
                                             2024-03    12.467119
                                             2024-04    12.018676
                                             2024-05    12.400586
                                             2024-06    12.168047
...                                                           ...
8. Outros artigos de uso pessoal e doméstico 2025-09     7.918904
                                             2025-10     8.699150
                                             2025-11    10.659018
                                             2025-12    11.465644
                                             2026-01     8.283442

[192 rows x 1 columns]

In [29]:
bu_agg_preds = Aggregator().fit_transform(bu_preds)
bu_agg_preds

indice_pond
Atividades                      Data                
1. Combustíveis e lubrificantes 2024-02    11.485453
                                2024-03    12.467119
                                2024-04    12.018676
                                2024-05    12.400586
                                2024-06    12.168047
...                                              ...
__total                         2025-09   107.259984
                                2025-10   111.573974
                                2025-11   115.931318
                                2025-12   135.580439
                                2026-01   111.306379

[216 rows x 1 columns]

In [30]:
metrics(test, bu_agg_preds, train).round(4)

,MAE,RMSE,MAPE,sMAPE,MASE
1. Combustíveis e lubrificantes,0.1739,0.1973,1.41,1.40,0.2331
"2. Hipermercados, supermercados, produtos alimentícios, bebidas e fumo",2.6160,3.3790,4.41,4.27,1.3873
"3. Tecidos, vestuário e calçados",0.2810,0.3896,4.16,4.22,0.5522
4. Móveis e eletrodomésticos,0.4039,0.4944,5.13,5.32,0.6532
"5. Artigos farmacêuticos, médicos, ortopédicos, de perfumaria e cosméticos",0.4073,0.4718,3.73,3.82,0.9988
"6. Livros, jornais, revistas e papelaria",0.0671,0.0755,25.65,30.87,0.8877
"7. Equipamentos e materiais para escritório, informática e comunicação",0.1082,0.1481,6.71,6.90,0.6197
8. Outros artigos de uso pessoal e doméstico,0.7002,0.7900,7.50,7.88,0.9819
__total,2.1257,2.6467,1.95,1.94,0.5700


## Top-Down → td_fcst: Forecast Proportions


In [31]:
td_fcst_pipe = TransformedTargetForecaster(steps=[
    # ('log', LogTransformer()),
    ('deseason', Deseasonalizer(sp=12)),
    ('diff', Differencer()),
    ('forecaster', ensemble),
    ('reconciler', TopdownReconciler(method='td_fcst'))

])

td_fcst_pipe.fit(train, fh=fh)
td_fcst_preds = td_fcst_pipe.predict()

td_fcst_preds

indice_pond
Atividades                      Data                
1. Combustíveis e lubrificantes 2024-02    11.485453
                                2024-03    12.467119
                                2024-04    12.018676
                                2024-05    12.400586
                                2024-06    12.168047
...                                              ...
__total                         2025-09   107.259984
                                2025-10   111.573974
                                2025-11   115.931318
                                2025-12   135.580439
                                2026-01   111.306379

[216 rows x 1 columns]

In [32]:
metrics(test, td_fcst_preds, train).round(4)

,MAE,RMSE,MAPE,sMAPE,MASE
1. Combustíveis e lubrificantes,0.1739,0.1973,1.41,1.40,0.2331
"2. Hipermercados, supermercados, produtos alimentícios, bebidas e fumo",2.6160,3.3790,4.41,4.27,1.3873
"3. Tecidos, vestuário e calçados",0.2810,0.3896,4.16,4.22,0.5522
4. Móveis e eletrodomésticos,0.4039,0.4944,5.13,5.32,0.6532
"5. Artigos farmacêuticos, médicos, ortopédicos, de perfumaria e cosméticos",0.4073,0.4718,3.73,3.82,0.9988
"6. Livros, jornais, revistas e papelaria",0.0671,0.0755,25.65,30.87,0.8877
"7. Equipamentos e materiais para escritório, informática e comunicação",0.1082,0.1481,6.71,6.90,0.6197
8. Outros artigos de uso pessoal e doméstico,0.7002,0.7900,7.50,7.88,0.9819
__total,2.1257,2.6467,1.95,1.94,0.5700


## Top-Down → td_share: Topdown Share

In [33]:
td_share_pipe = TransformedTargetForecaster(steps=[
    # ('log', LogTransformer()),
    ('deseason', Deseasonalizer(sp=12)),
    ('diff', Differencer()),
    ('forecaster', ensemble),
    ('reconciler', TopdownReconciler(method='td_share'))

])

td_share_pipe.fit(train, fh=fh)
td_share_preds = td_share_pipe.predict()

td_share_preds

indice_pond
Atividades                      Data                
1. Combustíveis e lubrificantes 2024-02     0.120988
                                2024-03     0.121403
                                2024-04     0.120555
                                2024-05     0.119834
                                2024-06     0.120707
...                                              ...
__total                         2025-09   107.259984
                                2025-10   111.573974
                                2025-11   115.931318
                                2025-12   135.580439
                                2026-01   111.306379

[216 rows x 1 columns]

In [34]:
metrics(test, td_share_preds, train).round(4)

,MAE,RMSE,MAPE,sMAPE,MASE
1. Combustíveis e lubrificantes,12.2676,12.2752,99.06,196.28,16.4432
"2. Hipermercados, supermercados, produtos alimentícios, bebidas e fumo",58.3587,58.4702,99.03,196.16,30.9480
"3. Tecidos, vestuário e calçados",6.1208,6.3061,99.09,196.38,12.0283
4. Móveis e eletrodomésticos,7.4960,7.5596,99.11,196.47,12.1225
"5. Artigos farmacêuticos, médicos, ortopédicos, de perfumaria e cosméticos",10.6617,10.6785,99.10,196.42,26.1427
"6. Livros, jornais, revistas e papelaria",0.2878,0.2998,99.26,197.07,3.8071
"7. Equipamentos e materiais para escritório, informática e comunicação",1.4897,1.5024,99.08,196.36,8.5316
8. Outros artigos de uso pessoal e doméstico,9.2862,9.3642,99.13,196.57,13.0222
__total,2.1257,2.6467,1.95,1.94,0.5700


# Optimal

In [59]:
opt_pipe = TransformedTargetForecaster(steps=[
    ('log', BoxCoxTransformer()),
    ('deseason', Deseasonalizer(sp=12)),
    ('diff', Differencer()),
    ('forecaster', ensemble),
    ('reconciler', OptimalReconciler())

])

opt_pipe.fit(train, fh=fh)
opt_preds = opt_pipe.predict()

display(opt_preds)
display( metrics(test, opt_preds, train).round(4) )

indice_pond
Atividades                      Data                
1. Combustíveis e lubrificantes 2024-02    11.430079
                                2024-03    12.483233
                                2024-04    11.971585
                                2024-05    12.404612
                                2024-06    12.166939
...                                              ...
__total                         2025-09   105.825348
                                2025-10   109.774957
                                2025-11   113.123598
                                2025-12   132.552905
                                2026-01   107.038077

[216 rows x 1 columns]

,MAE,RMSE,MAPE,sMAPE,MASE
1. Combustíveis e lubrificantes,0.1779,0.2095,1.44,1.44,0.2384
"2. Hipermercados, supermercados, produtos alimentícios, bebidas e fumo",1.8267,2.1505,3.11,3.05,0.9687
"3. Tecidos, vestuário e calçados",0.2466,0.3801,3.51,3.52,0.4846
4. Móveis e eletrodomésticos,0.5133,0.6135,6.54,6.83,0.8301
"5. Artigos farmacêuticos, médicos, ortopédicos, de perfumaria e cosméticos",0.3539,0.4199,3.24,3.32,0.8679
"6. Livros, jornais, revistas e papelaria",0.0225,0.0258,8.34,8.70,0.2982
"7. Equipamentos e materiais para escritório, informática e comunicação",0.1076,0.1334,7.00,6.88,0.6164
8. Outros artigos de uso pessoal e doméstico,0.6365,0.7324,6.84,7.17,0.8925
__total,1.7403,2.0151,1.60,1.60,0.4667


In [99]:
opt_pipe.forecaster_.forecasters_.loc['1. Combustíveis e lubrificantes'].values[0].get_fitted_params()['weights']

[np.int32(89), np.int32(59), np.int32(0), np.int32(38)]

In [102]:
train.columns
train.index.names

FrozenList(['Atividades', 'Data'])

In [133]:
# jpreds = pd.DataFrame(index=train.index.get_level_values('Data').unique(), 
#                       columns=train.index.get_level_values('Atividades').unique(),
#                       data=0)

# # jpreds = pd.DataFrame(index=pmc_agg.index.get_level_values('Data').unique(), 
#                     #   columns=pmc_agg.index.get_level_values('Atividades').unique(),
#                     #   data=0)


# for i, serie in enumerate(train.index.get_level_values('Atividades').unique()):
    
#     print(serie)
#     models = opt_pipe.forecaster_.forecasters_.loc[serie].values[0]
#     pesos_models = models.get_fitted_params()['weights']
#     # print([i/2 for i in pesos_models])

#     # display(len(models.forecasters_))

#     for j in range( len(models.forecasters_) ):

#         temp_forecaster = models.forecasters_[j][1]
#         # display(temp_forecaster)
        
#         # temp_pred = temp_forecaster.predict(fh=jpreds.index)
#         temp_pred = temp_forecaster.predict(fh=train.loc[serie].index)

#         # display(temp_pred)

#         temp_pred_weighted = temp_pred.multiply(pesos_models[j])
#         # display(temp_pred_weighted)

#         jpreds[serie] += temp_pred_weighted
#         # display(jpreds[[serie]])
#         # display(models.forecasters_[j][1].predict().multiply(pesos_models[j]/100).values.reshape(-1, 1))

# #     # .forecasters_[3][1].predict()#.get_fitted_params()
    
# jpreds

In [139]:
from pipeline.modelo import create_model, save_bundle

m1 = create_model()

m1.fit( pmc_agg.query("Data < '2025-10-01'") )

TransformedTargetForecaster(steps=[('forecaster',
                                    AutoEnsembleForecaster(forecasters=[StatsForecastAutoARIMA(sp=12),
                                                                        StatsForecastAutoETS(season_length=12),
                                                                        StatsForecastAutoCES(season_length=12),
                                                                        StatsForecastAutoTBATS(seasonal_periods=12)],
                                                           regressor=LGBMRegressor(verbosity=-1))),
                                   ('reconciler', OptimalReconciler())])

In [144]:
BUNDLE_PATH = r'../data/pmc_model_bundle.joblib'

# new_hist = pmc_agg.copy(deep=True)
# new_modelo = old_modelo.fit(new_hist)
# new_preds = new_modelo.predict(fh=FORECAST_HORIZON)
# new_full_data = pd.concat([new_hist, new_preds]).sort_index()

save_bundle(
              model = m1,
              hist = pmc_agg.query("Data < '2025-10-01'"),
              preds = m1.predict(fh=fh),
              # last_date = m1.cutoff,
              bundle_path= BUNDLE_PATH
            )

# Grid Search

In [37]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
pipe = TransformedTargetForecaster(steps=[
    # ('log', LogTransformer()),
    ('deseason', OptionalPassthrough(Deseasonalizer(sp=12))),
    ('diff', OptionalPassthrough(Differencer())),
    ('forecaster', ensemble),
    ('reconciler', BottomUpReconciler())

])


param_grid = {

'deseason__passthrough': [True, False],
'diff__passthrough': [True, False], 
# 'forecaster__regressor': [lgbm, ElasticNet(), LinearRegression()],
'reconciler': [TopdownReconciler(), OptimalReconciler()]
}


cv = TemporalTrainTestSplitter(test_size=24)

gscv = ForecastingGridSearchCV(
    forecaster=pipe,
    param_grid = param_grid, 
    cv=cv,
    error_score='raise'

)
gscv.fit(y=pmc_agg)

ValueError: y_pred and y_true do not have the same number of rows.